# 03. 데이터 분할 (Data Splitting)

이 노트북은 정제된 Seagate ST4000DM000 디스크 데이터를 머신러닝 학습 및 평가를 위한 데이터셋으로 분할합니다.

### 주요 분할 전략:
1. **시간 무관 그룹 층화 분할 (Group Stratified Split)**:
   - 고장 개체가 특정 시점(초기 불량 등)에 쏠려 있어 단순 관측 종료일 기준 분할 시 검증/테스트 셋에 고장 개체가 거의 남지 않는 **생존자 편향(Survivor Bias)**이 발생합니다.
   - 이를 해결하기 위해 시간 조건 대신 개체 단위의 무작위 층화 분할을 가합니다.
2. **물리적 개체 누수(Data Leakage) 원천 차단**:
   - 2단계 정제에서 2일 이상 공백으로 인해 분리되었던 개체(`serial_number_1`, `serial_number_2` 등)는 실제 동일한 물리적 하드디스크입니다.
   - 따라서 `regexp_replace(serial_number, '_[0-9]+$', '')` 처리를 통해 원래의 **Base Serial** 기준으로 통합 묶음(Family Binding) 처리한 후, 같은 물리 개체의 시계열은 100% 동일한 분할 폴드에 몰아서 배정합니다.
3. **4분할 구조 적용 (훈련:6, 튜닝:1, 보정:1, 평가:2)**:
   - **`train_raw` (60%)**: 학습용 (고장 표본 약 3,400개)
   - **`val_tune_raw` (10%)**: Optuna 하이퍼파라미터 튜닝용 (고장 표본 약 570개)
   - **`val_calib_raw` (10%)**: 독립 임계값(Threshold) 보정용 (고장 표본 약 570개, 튜닝 단계에서의 과적합 전이를 막아 실무 관제 오탐 폭발 차단)
   - **`test_raw` (20%)**: 최종 일반화 성능 검증용 (고장 표본 약 1,148개)

## 1. 환경 설정 및 데이터 로드

In [4]:
import duckdb
import os
import time
import json
import traceback
from pathlib import Path

input_parquet = "../data2/01_cleaned/ST4000DM000_cleaned_2.parquet"
output_dir = Path("../data2/03_splitting")
output_dir.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
PARQUET_COMPRESSION = "ZSTD"
DUCKDB_THREADS = 12
DUCKDB_MEMORY_LIMIT = "28GB"
DUCKDB_TEMP_DIR = output_dir / "_duckdb_tmp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"입력 데이터 경로: {input_parquet}")
print(f"출력 디렉토리: {output_dir.resolve()}")

# 주피터 세션 전체에서 유지될 전역 DuckDB 연결을 생성하고 PRAGMA 설정 적용
con = duckdb.connect()
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{str(DUCKDB_TEMP_DIR).replace("'", "''")}'")
print("✅ 전역 DuckDB 연결 생성 완료.")

입력 데이터 경로: ../data2/01_cleaned/ST4000DM000_cleaned_2.parquet
출력 디렉토리: C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting
✅ 전역 DuckDB 연결 생성 완료.


## 2. 물리 개체 통합 및 그룹 층화 분할 연산

DuckDB 해시 셔플 연산을 사용하여 Base Serial 단위로 6:1:1:2 분할을 생성합니다.

In [5]:
print("🚀 개체 그룹 층화 분할 연산 시작...")
start_time = time.time()

try:
    # 1. 물리적 개체 식별 및 고장 라벨 추출 (Base Serial 단위 통합)
    con.execute("""
        CREATE OR REPLACE TEMP TABLE entity_label AS
        SELECT
            regexp_replace(serial_number, '_[0-9]+$', '') AS entity_id,
            CAST(MAX(failure) AS BIGINT) AS entity_failed
        FROM read_parquet(?)
        GROUP BY 1
    """, [input_parquet])

    # 2. 고장 유무(entity_failed)별로 해시 셔플 및 정렬 순번 부여 (시드 고정)
    con.execute("""
        CREATE OR REPLACE TEMP TABLE entity_ranked AS
        SELECT
            entity_id,
            entity_failed,
            ROW_NUMBER() OVER (
                PARTITION BY entity_failed
                ORDER BY hash(entity_id || '|' || CAST(? AS VARCHAR))
            ) AS rn,
            COUNT(*) OVER (PARTITION BY entity_failed) AS grp_n
        FROM entity_label
    """, [RANDOM_SEED])

    # 3. 층화 분할 배정 (6:1:1:2)
    con.execute("""
        CREATE OR REPLACE TEMP TABLE entity_split AS
        WITH c AS (
            SELECT
                entity_id,
                entity_failed,
                rn,
                grp_n,
                CAST(FLOOR(grp_n * 0.6) AS BIGINT) AS c1,
                CAST(FLOOR(grp_n * 0.1) AS BIGINT) AS c2,
                CAST(FLOOR(grp_n * 0.1) AS BIGINT) AS c3
            FROM entity_ranked
        )
        SELECT
            entity_id,
            entity_failed,
            CASE
                WHEN rn <= c1 THEN 'train_raw'
                WHEN rn <= c1 + c2 THEN 'val_tune_raw'
                WHEN rn <= c1 + c2 + c3 THEN 'val_calib_raw'
                ELSE 'test_raw'
            END AS split
        FROM c
    """)
    
    print(f"✅ 분할 테이블 설계 완료! (소요 시간: {time.time() - start_time:.2f}초)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

🚀 개체 그룹 층화 분할 연산 시작...
✅ 분할 테이블 설계 완료! (소요 시간: 0.13초)


## 3. 분할 결과 통계 및 분포 출력

In [6]:
try:
    # 개체(Entity) 기준 요약
    entity_summary = con.execute("""
        SELECT
            split,
            COUNT(*) AS total_entities,
            SUM(entity_failed) AS failed_entities,
            ROUND(SUM(entity_failed) * 1.0 / COUNT(*), 6) AS failed_ratio
        FROM entity_split
        GROUP BY split
        ORDER BY split
    """).fetchdf()
    
    # 로우(Row) 기준 요약 (실제 시계열 데이터 행 수)
    row_summary = con.execute("""
        SELECT
            e.split,
            COUNT(*) AS total_rows,
            SUM(CAST(d.failure AS BIGINT)) AS failed_rows,
            ROUND(SUM(CAST(d.failure AS DOUBLE)) / COUNT(*), 6) AS failed_row_ratio
        FROM read_parquet(?) d
        JOIN entity_split e
          ON regexp_replace(d.serial_number, '_[0-9]+$', '') = e.entity_id
        GROUP BY e.split
        ORDER BY e.split
    """, [input_parquet]).fetchdf()
    
    print("[개체 기준 분할 요약]")
    print(entity_summary.to_string(index=False))
    
    print("\n[row 기준 분할 요약(참고)]")
    print(row_summary.to_string(index=False))
except Exception as e:
    print(f"❌ 오류 발생: {e}")

[개체 기준 분할 요약]
        split  total_entities  failed_entities  failed_ratio
     test_raw            7387           1141.0      0.154461
    train_raw           22158           3420.0      0.154346
val_calib_raw            3693            570.0      0.154346
 val_tune_raw            3693            570.0      0.154346

[row 기준 분할 요약(참고)]
        split  total_rows  failed_rows  failed_row_ratio
     test_raw    15423145      34726.0          0.002252
    train_raw    46461541     103817.0          0.002234
val_calib_raw     7695141      17355.0          0.002255
 val_tune_raw     7783151      17416.0          0.002238


## 4. 데이터셋 저장 (Save Split Files)

분할 쿼리를 실행하여 실제 물리 파티션을 Parquet 형식으로 내보냅니다.

In [7]:
print("🚀 분할 데이터셋 물리 파일 내보내기 시작...")
start_time = time.time()

try:
    splits = ['train_raw', 'val_tune_raw', 'val_calib_raw', 'test_raw']
    for idx, sp in enumerate(splits, start=1):
        out_path = (output_dir / f"{sp}.parquet").as_posix()
        tmp_path = (output_dir / f".{sp}.tmp.parquet").as_posix()
        
        print(f"  [{idx}/4] 저장 중: {sp} -> {out_path} ...")
        
        con.execute(f"""
            COPY (
                SELECT d.*
                FROM read_parquet('{input_parquet}') d
                JOIN entity_split e
                  ON regexp_replace(d.serial_number, '_[0-9]+$', '') = e.entity_id
                WHERE e.split = '{sp}'
            ) TO '{tmp_path}' (FORMAT PARQUET, COMPRESSION {PARQUET_COMPRESSION})
        """)
        
        if os.path.exists(out_path):
            os.remove(out_path)
        os.replace(tmp_path, out_path)
        
        print(f"  -> 완료! ({out_path})")
    print(f"✅ 모든 데이터셋 분리 및 저장 완료! (소요 시간: {time.time() - start_time:.2f}초)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

🚀 분할 데이터셋 물리 파일 내보내기 시작...
  [1/4] 저장 중: train_raw -> ../data2/03_splitting/train_raw.parquet ...
  -> 완료! (../data2/03_splitting/train_raw.parquet)
  [2/4] 저장 중: val_tune_raw -> ../data2/03_splitting/val_tune_raw.parquet ...
  -> 완료! (../data2/03_splitting/val_tune_raw.parquet)
  [3/4] 저장 중: val_calib_raw -> ../data2/03_splitting/val_calib_raw.parquet ...
  -> 완료! (../data2/03_splitting/val_calib_raw.parquet)
  [4/4] 저장 중: test_raw -> ../data2/03_splitting/test_raw.parquet ...
  -> 완료! (../data2/03_splitting/test_raw.parquet)
✅ 모든 데이터셋 분리 및 저장 완료! (소요 시간: 10.66초)


## 5. 엄밀한 데이터 분할 무결성 검증 테스트 (Verification Tests)

저장된 4개 분할 데이터셋의 로우 수, 유니크 개체 수, 고장 개체 수의 총합이 원본 데이터셋과 완벽히 일치하는지, 개체 간 누수(Overlap)가 전혀 없는지 수학적으로 엄밀히 입증합니다.

In [8]:
print("🔍 [분할 무결성 검증] 정밀 테스트 시작...")

try:
    splits = ['train_raw', 'val_tune_raw', 'val_calib_raw', 'test_raw']
    split_files = [output_dir / f"{s}.parquet" for s in splits]
    split_paths = [sf.as_posix() for sf in split_files]
    
    # 1. 파일 셋 존재 확인
    print("Test 1: 분할 파일 물리 생성 테스트")
    for sf in split_files:
        assert sf.exists(), f"오류: 분할 파일 {sf.name}이 생략되었습니다"
    print("  -> [PASS] 4개 분할 파일 정상 생성 확인.")
    
    # 2. 총 Row 수 및 고장 Row 수 통합성 검증 (원본 vs 분할별 합산)
    print("Test 2: 총 Row 수 및 고장 Row 합산이 일치 테스트")
    orig_rows, orig_failures = con.execute(
        "SELECT COUNT(*), SUM(CAST(failure AS BIGINT)) FROM read_parquet(?)", [input_parquet]
    ).fetchone()
    
    split_rows = sum([con.execute("SELECT COUNT(*) FROM read_parquet(?)", [p]).fetchone()[0] for p in split_paths])
    split_failures = sum([con.execute("SELECT SUM(CAST(failure AS BIGINT)) FROM read_parquet(?)", [p]).fetchone()[0] for p in split_paths])
    
    assert orig_rows == split_rows, f"오류: 총 Row 수 불일치! 원본={orig_rows:,}, 분할별={split_rows:,}"
    assert orig_failures == split_failures, f"오류: 총 고장 Row 수 불일치! 원본={orig_failures:,}, 분할별={split_failures:,}"
    print(f"  -> [PASS] Row 수 {split_rows:,}건 및 고장 Row 수 {split_failures:,}건 완전 일치.")
    
    # 3. 유니크 물리 개체(Base Serial) 수 합산 검증
    print("Test 3: 유니크 물리 개체(Base Serial) 합산이 일치 테스트")
    orig_entities = con.execute(
        "SELECT COUNT(DISTINCT regexp_replace(serial_number, '_[0-9]+$', '')) FROM read_parquet(?)", [input_parquet]
    ).fetchone()[0]
    
    split_entities_list = [con.execute(
        "SELECT DISTINCT regexp_replace(serial_number, '_[0-9]+$', '') as bs FROM read_parquet(?)", [p]
    ).df()['bs'].tolist() for p in split_paths]
    
    total_split_entities = sum([len(s) for s in split_entities_list])
    assert orig_entities == total_split_entities, f"오류: 개체 수 불일치! 원본={orig_entities:,}, 분할별={total_split_entities:,}"
    print(f"  -> [PASS] 유니크 물리 개체 수 {total_split_entities:,}개 완전 일치.")
    
    # 4. 개체 간 교집합 배정/침수(Overlap) 무원칙 침수 차단 검증
    print("Test 4: 데이터셋 간 물리 개체 침수(Data Leakage) 존재 여부 검증")
    sets = [set(s) for s in split_entities_list]
    
    pairs = [
        ('train', 'val_tune', sets[0], sets[1]),
        ('train', 'val_calib', sets[0], sets[2]),
        ('train', 'test', sets[0], sets[3]),
        ('val_tune', 'val_calib', sets[1], sets[2]),
        ('val_tune', 'test', sets[1], sets[3]),
        ('val_calib', 'test', sets[2], sets[3])
    ]
    
    for name1, name2, s1, s2 in pairs:
        overlap = s1.intersection(s2)
        assert len(overlap) == 0, f"오류: {name1}과 {name2} 데이터셋 사이에 {len(overlap)}개의 물리 개체 침수가 발생했습니다!"
        
    print("  -> [PASS] 데이터셋 간 100% 완역한 물리 개체 경계 완료 (침수=0).")
    
    # 5. 스키마 동일성 검증
    print("Test 5: 데이터셋별 스키마(Schema) 동일성 테스트")
    orig_schema = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [input_parquet]).fetchdf()
    orig_cols = orig_schema['column_name'].tolist()
    orig_types = orig_schema['column_type'].tolist()
    
    for p in split_paths:
        schema = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [p]).fetchdf()
        assert schema['column_name'].tolist() == orig_cols, f"오류: {p}의 컬럼 목록이 일치하지 않습니다!"
        assert schema['column_type'].tolist() == orig_types, f"오류: {p}의 데이터 타입 목록이 일치하지 않습니다!"
    print("  -> [PASS] 모든 분할 파일 스키마 완전 일치.")

    # ── 강화 6: 각 split 비어있지 않은지 검증 ──
    print("Test 6: 각 분할 파일 row 수 > 0 검증")
    for s, p in zip(splits, split_paths):
        cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [p]).fetchone()[0]
        assert cnt > 0, f"오류: {s} 분할 파일이 비어 있습니다"
    print("  -> [PASS] 모든 분할 파일에 데이터 존재 확인.")

    # ── 강화 7: failure 값 범위 검증 ──
    print("Test 7: 모든 분할 파일 failure 값 범위 {0,1} 검증")
    for s, p in zip(splits, split_paths):
        fv = con.execute("SELECT DISTINCT failure FROM read_parquet(?) ORDER BY failure", [p]).fetchall()
        fset = set(r[0] for r in fv)
        assert fset.issubset({0, 1}), f"오류: {s}에서 failure 값 {fset}이 발견됨"
    print("  -> [PASS] 모든 분할 파일 failure 값 {0,1} 확인.")

    # ── 강화 8: serial_number 내 date 정렬 연속성 검증 ──
    print("Test 8: serial_number 내 date 연속성 (gap<=1일) 검증")
    for s, p in zip(splits, split_paths):
        gaps = con.execute(f"""
            WITH Lagged AS (
                SELECT serial_number, date,
                       LAG(date) OVER (PARTITION BY serial_number ORDER BY date) as prev_date
                FROM read_parquet('{p}')
            )
            SELECT COUNT(*) FROM Lagged
            WHERE prev_date IS NOT NULL 
              AND date_diff('day', CAST(prev_date AS DATE), CAST(date AS DATE)) >= 3
        """).fetchone()[0]
        assert gaps == 0, f"오류: {s}에서 3일 이상 gap이 {gaps}건 발견됨"
    print("  -> [PASS] 모든 분할 파일 내 시계열 연속성 확인.")
    
    print("\n✅ [분할 무결성 검증 완료] 모든 정밀 테스트를 만족합니다 (8/8 PASS)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")
finally:
    con.close()
    if DUCKDB_TEMP_DIR.exists():
        import shutil
        shutil.rmtree(DUCKDB_TEMP_DIR, ignore_errors=True)


🔍 [분할 무결성 검증] 정밀 테스트 시작...
Test 1: 분할 파일 물리 생성 테스트
  -> [PASS] 4개 분할 파일 정상 생성 확인.
Test 2: 총 Row 수 및 고장 Row 합산이 일치 테스트
  -> [PASS] Row 수 77,362,978건 및 고장 Row 수 173,314건 완전 일치.
Test 3: 유니크 물리 개체(Base Serial) 합산이 일치 테스트
  -> [PASS] 유니크 물리 개체 수 36,931개 완전 일치.
Test 4: 데이터셋 간 물리 개체 침수(Data Leakage) 존재 여부 검증
  -> [PASS] 데이터셋 간 100% 완역한 물리 개체 경계 완료 (침수=0).
Test 5: 데이터셋별 스키마(Schema) 동일성 테스트
  -> [PASS] 모든 분할 파일 스키마 완전 일치.
Test 6: 각 분할 파일 row 수 > 0 검증
  -> [PASS] 모든 분할 파일에 데이터 존재 확인.
Test 7: 모든 분할 파일 failure 값 범위 {0,1} 검증
  -> [PASS] 모든 분할 파일 failure 값 {0,1} 확인.
Test 8: serial_number 내 date 연속성 (gap<=1일) 검증
  -> [PASS] 모든 분할 파일 내 시계열 연속성 확인.

✅ [분할 무결성 검증 완료] 모든 정밀 테스트를 만족합니다 (8/8 PASS)
